## Upload a Custom Random Image Dataset to S3

### Objective
Create a synthetic image dataset with 100 random images, upload every image to S3 through LAILA, and store the `{name: global_id}` mapping as a manifest entry named `my_dataset_manifest`.


In [ ]:
%load_ext autoreload
%autoreload 2

### Objective
Import dependencies, configure paths, and define the constants used by the example.


In [ ]:
import numpy as np
from PIL import Image

import laila
from laila.pool import S3Pool

POOL_NICKNAME = "custom_image_s3_pool"
MANIFEST_NICKNAME = "my_dataset_manifest"
IMAGE_COUNT = 100
IMAGE_SIZE = (64, 64)


### Objective
Define the helper for constructing the S3 pool used by this example.


In [ ]:
# These argument names are arbitrary and can be changed by the user.
# laila.args is just a container for user-provided arguments.
# Replace these placeholders with your own AWS credentials before running.
laila.args.AWS_BUCKET_NAME = "your-bucket-name"
laila.args.AWS_ACCESS_KEY_ID = "your-access-key-id"
laila.args.AWS_SECRET_ACCESS_KEY = "your-secret-access-key"
laila.args.AWS_REGION = "us-east-1"


def build_s3_pool() -> S3Pool:
    return S3Pool(
        bucket_name=laila.args.AWS_BUCKET_NAME,
        access_key_id=laila.args.AWS_ACCESS_KEY_ID,
        secret_access_key=laila.args.AWS_SECRET_ACCESS_KEY,
        region_name=laila.args.AWS_REGION,
    )

s3_pool = build_s3_pool()
laila.add_pool(s3_pool, pool_nickname=POOL_NICKNAME)


### Objective
Register the S3 pool, generate 100 random RGB images with deterministic nicknames, and build the `{name: global_id}` manifest dictionary.


In [ ]:
rng = np.random.default_rng(42)
image_entries = []
manifest_dict = {}

for j in range(IMAGE_COUNT):
    image_name = f"image_{j}"
    image_array = rng.integers(0, 256, size=(*IMAGE_SIZE, 3), dtype=np.uint8)

    # Validate the array as an actual image before wrapping it with LAILA.
    new_image = Image.fromarray(image_array, mode="RGB")

    image_entry = laila.constant(data=new_image, nickname=image_name)
    image_entries.append(image_entry)
    manifest_dict[image_name] = image_entry.global_id

print("Generated image entries:", len(image_entries))
print("First item:", next(iter(manifest_dict.items())))

### Objective
Upload the full image dataset to S3 through one batched LAILA call.


In [ ]:
upload_group_future = laila.memorize(image_entries, pool_nickname=POOL_NICKNAME)
print("Upload future type:", type(upload_group_future).__name__)
print("Status before wait:", upload_group_future.status)

upload_group_future.wait()
print("Status after wait:", upload_group_future.status)
print("Uploaded entries:", len(upload_group_future))

### Objective
Persist the `{name: global_id}` manifest itself as a deterministic LAILA entry named `my_dataset_manifest`.


In [ ]:
manifest_entry = laila.constant(data=manifest_dict, nickname=MANIFEST_NICKNAME)
manifest_future = laila.memorize(manifest_entry, pool_nickname=POOL_NICKNAME)
manifest_future.wait()

print("Manifest nickname:", MANIFEST_NICKNAME)
print("Manifest global_id:", manifest_entry.global_id)
print("Manifest size:", len(manifest_dict))
print("Sample manifest rows:")
for index, item in enumerate(manifest_dict.items()):
    if index >= 5:
        break
    print(item)